In [ ]:
# Cell 1 — Install
!pip install yfinance -q

In [ ]:
# Cell 2 — Mount Drive
from google.colab import drive
drive.mount('/content/drive')
import os
OUTPUT_DIR = '/content/drive/MyDrive/ethical-finance/ohlcv_backfill'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Drive monté —', OUTPUT_DIR)

In [ ]:
# Cell 3 — Config
from datetime import date
START = '2026-05-15'
END = str(date.today())
print(f'Backfill {START} → {END}')

In [ ]:
# Cell 4 — Tickers depuis l'API (tous SP500+CAC40+ETF)
import requests

API_URL = 'https://api.sauhabah-advisory.eu'

# Récupère tous les tickers depuis la DB
r = requests.get(f'{API_URL}/api/tickers?universe=sp500&limit=600')
sp500 = [t['ticker'] for t in r.json().get('tickers', [])]

r2 = requests.get(f'{API_URL}/api/tickers?universe=cac40&limit=100')
cac40 = [t['ticker'] for t in r2.json().get('tickers', [])]

r3 = requests.get(f'{API_URL}/api/tickers?universe=etf_broad&limit=50')
etf_broad = [t['ticker'] for t in r3.json().get('tickers', [])]

r4 = requests.get(f'{API_URL}/api/tickers?universe=etf_precious_metals&limit=50')
etf_pm = [t['ticker'] for t in r4.json().get('tickers', [])]

# Indices
indices = ['^GSPC', '^FCHI', '^GDAXI', '^VIX', '^N225', '^FTSE']

TICKERS = list(set(sp500 + cac40 + etf_broad + etf_pm + indices))
print(f'{len(TICKERS)} tickers total')
print(f'SP500: {len(sp500)}, CAC40: {len(cac40)}, ETF: {len(etf_broad)+len(etf_pm)}')

In [ ]:
# Cell 5 — Download + save CSV par batch
import yfinance as yf
import pandas as pd
import time

all_dfs = []
errors = []

for i, ticker in enumerate(TICKERS):
    try:
        df = yf.download(ticker, start=START, end=END, progress=False, auto_adjust=True)
        if df.empty:
            errors.append(ticker)
            continue
        df = df.reset_index()
        df['ticker'] = ticker
        df = df.rename(columns={'Date':'date','Open':'open','High':'high','Low':'low','Close':'close','Volume':'volume'})
        df['adj_close'] = df['close']
        df = df[['ticker','date','open','high','low','close','adj_close','volume']]
        all_dfs.append(df)
        if i % 20 == 0:
            print(f'{i}/{len(TICKERS)} — {ticker} — {len(df)} rows')
    except Exception as e:
        errors.append(f'{ticker}: {e}')
    time.sleep(0.3)

result = pd.concat(all_dfs, ignore_index=True)
print(f'Total: {len(result)} rows — {result.ticker.nunique()} tickers')
print(f'Errors: {errors}')

In [ ]:
# Cell 6 — Save to Drive
import datetime
filename = f'ohlcv_backfill_{datetime.date.today()}.csv'
path = f'{OUTPUT_DIR}/{filename}'
result.to_csv(path, index=False)
print(f'Saved: {path}')
print(result.head())